In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"D:\PROJECTS\customer_demand_analysis\data")

for file in DATA_DIR.glob("*.csv"):
    print(f"{file.name:35} {file.stat().st_size / (1024**2):.2f} MB")


aisles.csv                          0.00 MB
departments.csv                     0.00 MB
orders.csv                          103.92 MB
order_products__prior.csv           550.80 MB
order_products__train.csv           23.54 MB
products.csv                        2.07 MB


In [3]:
products = pd.read_csv(DATA_DIR / "products.csv")
aisles = pd.read_csv(DATA_DIR / "aisles.csv")
departments = pd.read_csv(DATA_DIR / "departments.csv")

print("Products:")
print(products.head())
print(products.shape)

print("\nAisles:")
print(aisles.head())
print(aisles.shape)

print("\nDepartments:")
print(departments.head())
print(departments.shape)

Products:
   product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4           5                          Green Chile Anytime Sauce         5   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13  
(49688, 4)

Aisles:
   aisle_id                       aisle
0         1       prepared soups salads
1         2           specialty cheeses
2         3         energy granola bars
3         4               instant foods
4         5  marinades meat preparation
(134, 2)

Departments:
   department_id department
0              1     frozen
1              2      other
2              3     bakery
3           

In [4]:
orders = pd.read_csv(DATA_DIR / "orders.csv")

print(orders.head())
print(orders.shape)
print(orders.dtypes)

   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  
(3421083, 7)
order_id                    int64
user_id                     int64
eval_set                      str
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object


In [5]:
prior_sample = pd.read_csv(
    DATA_DIR / "order_products__prior.csv",
    nrows=10000
)

print(prior_sample.head())
print(prior_sample.shape)
print(prior_sample.dtypes)

   order_id  product_id  add_to_cart_order  reordered
0         2       33120                  1          1
1         2       28985                  2          1
2         2        9327                  3          0
3         2       45918                  4          1
4         2       30035                  5          0
(10000, 4)
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object


In [6]:
train = pd.read_csv(
    DATA_DIR / "order_products__train.csv"
)

print(train.head())
print(train.shape)
print(train.dtypes)

   order_id  product_id  add_to_cart_order  reordered
0         1       49302                  1          1
1         1       11109                  2          1
2         1       10246                  3          0
3         1       49683                  4          0
4         1       43633                  5          1
(1384617, 4)
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object


## Feature Engineering Plan

The feature pipeline will use historical customer purchasing behavior from
the prior orders to create customer-level, product-level,
customer-product interaction, and category-affinity features.

Features will be constructed using only information available before
the future order to prevent data leakage.

The large order-product history will be processed in chunks to improve
memory efficiency and scalability.

In [7]:
#Prepare historical order data

# Keep only the columns needed for feature engineering
order_cols = [
    "order_id",
    "user_id",
    "eval_set",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order"
]

orders_hist = orders[order_cols].copy()

# Keep only historical (prior) orders for feature generation
orders_hist = orders_hist[orders_hist["eval_set"] == "prior"].copy()

print("Historical orders shape:", orders_hist.shape)
print("\nColumns:")
print(orders_hist.columns.tolist())

print("\nEvaluation sets:")
print(orders_hist["eval_set"].value_counts())

print("\nMissing values:")
print(orders_hist.isna().sum())

Historical orders shape: (3214874, 7)

Columns:
['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

Evaluation sets:
eval_set
prior    3214874
Name: count, dtype: int64

Missing values:
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64


In [8]:
# Step 2.2 — Create customer-level order features

customer_features = (
    orders_hist
    .groupby("user_id")
    .agg(
        user_total_orders=("order_id", "count"),
        user_avg_days_between_orders=("days_since_prior_order", "mean"),
        user_avg_order_hour=("order_hour_of_day", "mean"),
        user_avg_order_dow=("order_dow", "mean")
    )
    .reset_index()
)

print("Customer features shape:", customer_features.shape)
print("\nCustomer features:")
print(customer_features.head())

print("\nMissing values:")
print(customer_features.isna().sum())

Customer features shape: (206209, 5)

Customer features:
   user_id  user_total_orders  user_avg_days_between_orders  \
0        1                 10                     19.555556   
1        2                 14                     15.230769   
2        3                 12                     12.090909   
3        4                  5                     13.750000   
4        5                  4                     13.333333   

   user_avg_order_hour  user_avg_order_dow  
0            10.300000            2.500000  
1            10.571429            2.142857  
2            16.416667            1.083333  
3            12.600000            4.800000  
4            16.000000            1.750000  

Missing values:
user_id                         0
user_total_orders               0
user_avg_days_between_orders    0
user_avg_order_hour             0
user_avg_order_dow              0
dtype: int64


## Step 2.3 — Process Historical Product Purchases

The prior-order product history contains the historical products purchased by each customer. Because this file is large, it will be processed in chunks to reduce memory usage.

The resulting information will be used to create customer-product, product-level, and purchasing-behavior features.

In [9]:
# Step 2.3 — Inspect the prior purchase data in chunks

PRIOR_FILE = DATA_DIR / "order_products__prior.csv"

# Read only the columns required for feature engineering
prior_cols = [
    "order_id",
    "product_id",
    "add_to_cart_order",
    "reordered"
]

# Read the first chunk only to verify the chunking approach
prior_chunk = pd.read_csv(
    PRIOR_FILE,
    usecols=prior_cols,
    nrows=100_000
)

print("First chunk shape:", prior_chunk.shape)
print("\nColumns:")
print(prior_chunk.columns.tolist())

print("\nData types:")
print(prior_chunk.dtypes)

print("\nMissing values:")
print(prior_chunk.isna().sum())

print("\nReordered distribution:")
print(prior_chunk["reordered"].value_counts())

First chunk shape: (100000, 4)

Columns:
['order_id', 'product_id', 'add_to_cart_order', 'reordered']

Data types:
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object

Missing values:
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

Reordered distribution:
reordered
1    59514
0    40486
Name: count, dtype: int64


## Step 2.4 — Build Customer-Product Purchase History

The prior product-order data is joined with historical order information using `order_id` so that each product purchase can be associated with a customer.

Customer-product interaction features will be derived from historical purchases, including purchase count, reorder count, reorder rate, last purchase order, and average cart position.

In [10]:
# Step 2.4 — Build customer-product historical aggregates

from collections import defaultdict

# Map order_id -> user/order information
order_lookup = orders_hist[
    [
        "order_id",
        "user_id",
        "order_number"
    ]
].copy()

# Use a dictionary for fast lookup during chunk processing
order_lookup = order_lookup.set_index("order_id")

# Storage for aggregated customer-product statistics
customer_product_stats = defaultdict(
    lambda: {
        "purchase_count": 0,
        "reorder_count": 0,
        "last_order_number": 0,
        "cart_position_sum": 0
    }
)

chunk_size = 500_000
processed_rows = 0

for chunk in pd.read_csv(
    PRIOR_FILE,
    usecols=prior_cols,
    chunksize=chunk_size
):
    # Attach customer/order information
    chunk = chunk.join(
        order_lookup,
        on="order_id",
        how="inner"
    )

    # Aggregate within the current chunk
    grouped = (
        chunk
        .groupby(["user_id", "product_id"])
        .agg(
            purchase_count=("order_id", "count"),
            reorder_count=("reordered", "sum"),
            last_order_number=("order_number", "max"),
            cart_position_sum=("add_to_cart_order", "sum")
        )
    )

    # Add chunk results to global aggregates
    for (user_id, product_id), row in grouped.iterrows():
        key = (user_id, product_id)

        stats = customer_product_stats[key]

        stats["purchase_count"] += int(row["purchase_count"])
        stats["reorder_count"] += int(row["reorder_count"])
        stats["last_order_number"] = max(
            stats["last_order_number"],
            int(row["last_order_number"])
        )
        stats["cart_position_sum"] += int(row["cart_position_sum"])

    processed_rows += len(chunk)

    print(f"Processed {processed_rows:,} rows")

print("\nTotal customer-product pairs:", len(customer_product_stats))

Processed 500,000 rows
Processed 1,000,000 rows
Processed 1,500,000 rows
Processed 2,000,000 rows
Processed 2,500,000 rows
Processed 3,000,000 rows
Processed 3,500,000 rows
Processed 4,000,000 rows
Processed 4,500,000 rows
Processed 5,000,000 rows
Processed 5,500,000 rows
Processed 6,000,000 rows
Processed 6,500,000 rows
Processed 7,000,000 rows
Processed 7,500,000 rows
Processed 8,000,000 rows
Processed 8,500,000 rows
Processed 9,000,000 rows
Processed 9,500,000 rows
Processed 10,000,000 rows
Processed 10,500,000 rows
Processed 11,000,000 rows
Processed 11,500,000 rows
Processed 12,000,000 rows
Processed 12,500,000 rows
Processed 13,000,000 rows
Processed 13,500,000 rows
Processed 14,000,000 rows
Processed 14,500,000 rows
Processed 15,000,000 rows
Processed 15,500,000 rows
Processed 16,000,000 rows
Processed 16,500,000 rows
Processed 17,000,000 rows
Processed 17,500,000 rows
Processed 18,000,000 rows
Processed 18,500,000 rows
Processed 19,000,000 rows
Processed 19,500,000 rows
Process

## Step 2.5 — Create Customer-Product Features

The aggregated historical customer-product statistics are converted into model-ready features. These features capture purchase frequency, reorder behavior, recency-related order position, and typical cart position for each customer-product pair.

In [11]:
# Step 2.5 — Create customer-product feature DataFrame

customer_product_features = pd.DataFrame.from_dict(
    customer_product_stats,
    orient="index"
)

# Convert the tuple index (user_id, product_id) into columns
customer_product_features.index = pd.MultiIndex.from_tuples(
    customer_product_features.index,
    names=["user_id", "product_id"]
)

customer_product_features = customer_product_features.reset_index()

# Rename columns to ML-friendly feature names
customer_product_features = customer_product_features.rename(
    columns={
        "purchase_count": "user_product_purchase_count",
        "reorder_count": "user_product_reorder_count",
        "last_order_number": "user_product_last_order_number",
        "cart_position_sum": "user_product_cart_position_sum"
    }
)

# Calculate reorder rate
customer_product_features["user_product_reorder_rate"] = (
    customer_product_features["user_product_reorder_count"]
    / customer_product_features["user_product_purchase_count"]
)

# Calculate average cart position
customer_product_features["user_product_avg_cart_position"] = (
    customer_product_features["user_product_cart_position_sum"]
    / customer_product_features["user_product_purchase_count"]
)

# Remove intermediate column
customer_product_features = customer_product_features.drop(
    columns=["user_product_cart_position_sum"]
)

print("Customer-product feature shape:")
print(customer_product_features.shape)

print("\nFeatures:")
print(customer_product_features.head())

print("\nMissing values:")
print(customer_product_features.isna().sum())

print("\nData types:")
print(customer_product_features.dtypes)

Customer-product feature shape:
(13307953, 7)

Features:
   user_id  product_id  user_product_purchase_count  \
0        7        4920                            7   
1        7        4945                            3   
2        7        8277                            3   
3        7       11520                            1   
4        7       13198                            8   

   user_product_reorder_count  user_product_last_order_number  \
0                           6                              17   
1                           2                              18   
2                           2                              17   
3                           0                              17   
4                           7                              20   

   user_product_reorder_rate  user_product_avg_cart_position  
0                   0.857143                        4.714286  
1                   0.666667                       11.333333  
2                   0.666667    

## Step 2.6 — Add Customer-Product Recency

Customer-product recency is represented as the number of customer orders since the product was last purchased. This is calculated from the customer's total historical orders and the last order number in which the product appeared.

This feature captures how recently a product was relevant to the customer's purchasing behavior without introducing future-order information.

In [12]:
# Step 2.6 — Calculate customer-product recency

# Get the latest historical order number for each customer
user_order_summary = (
    orders_hist
    .groupby("user_id")
    .agg(
        user_last_order_number=("order_number", "max")
    )
    .reset_index()
)

# Add customer's latest historical order number
customer_product_features = customer_product_features.merge(
    user_order_summary,
    on="user_id",
    how="left"
)

# Number of customer orders since the product was last purchased
customer_product_features["user_product_recency_orders"] = (
    customer_product_features["user_last_order_number"]
    - customer_product_features["user_product_last_order_number"]
)

# Remove intermediate column
customer_product_features = customer_product_features.drop(
    columns=["user_last_order_number"]
)

print("Customer-product feature shape:")
print(customer_product_features.shape)

print("\nRecency statistics:")
print(
    customer_product_features["user_product_recency_orders"]
    .describe()
)

print("\nMissing values:")
print(
    customer_product_features[
        ["user_product_recency_orders"]
    ].isna().sum()
)

print("\nSample:")
print(
    customer_product_features[
        [
            "user_id",
            "product_id",
            "user_product_purchase_count",
            "user_product_reorder_rate",
            "user_product_recency_orders"
        ]
    ].head()
)

Customer-product feature shape:
(13307953, 8)

Recency statistics:
count    1.330795e+07
mean     9.512767e+00
std      1.341695e+01
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      1.200000e+01
max      9.800000e+01
Name: user_product_recency_orders, dtype: float64

Missing values:
user_product_recency_orders    0
dtype: int64

Sample:
   user_id  product_id  user_product_purchase_count  \
0        7        4920                            7   
1        7        4945                            3   
2        7        8277                            3   
3        7       11520                            1   
4        7       13198                            8   

   user_product_reorder_rate  user_product_recency_orders  
0                   0.857143                            3  
1                   0.666667                            2  
2                   0.666667                            3  
3                   0.000000                            3  
4   

## Step 2.7 — Create Product-Level Features

Product-level features summarize historical purchasing behavior across all customers. These features capture overall product popularity, customer reach, and reorder behavior.

The features will be derived only from prior-order history so that future purchase outcomes are not used as model inputs.

In [13]:
# Step 2.7 — Create product-level features

product_features = (
    customer_product_features
    .groupby("product_id")
    .agg(
        product_total_purchases=("user_product_purchase_count", "sum"),
        product_unique_users=("user_id", "nunique"),
        product_total_reorders=("user_product_reorder_count", "sum")
    )
    .reset_index()
)

# Calculate overall product reorder rate
product_features["product_reorder_rate"] = (
    product_features["product_total_reorders"]
    / product_features["product_total_purchases"]
)

# Remove intermediate aggregation
product_features = product_features.drop(
    columns=["product_total_reorders"]
)

print("Product features shape:")
print(product_features.shape)

print("\nProduct features:")
print(product_features.head())

print("\nMissing values:")
print(product_features.isna().sum())

print("\nProduct reorder rate statistics:")
print(product_features["product_reorder_rate"].describe())

Product features shape:
(49677, 4)

Product features:
   product_id  product_total_purchases  product_unique_users  \
0           1                     1852                   716   
1           2                       90                    78   
2           3                      277                    74   
3           4                      329                   182   
4           5                       15                     6   

   product_reorder_rate  
0              0.613391  
1              0.133333  
2              0.732852  
3              0.446809  
4              0.600000  

Missing values:
product_id                 0
product_total_purchases    0
product_unique_users       0
product_reorder_rate       0
dtype: int64

Product reorder rate statistics:
count    49677.000000
mean         0.366461
std          0.208103
min          0.000000
25%          0.208075
50%          0.376623
75%          0.529307
max          0.941176
Name: product_reorder_rate, dtype: float64


## Step 2.8 — Add Product Category Information

Product metadata is joined with the engineered product features to identify the aisle and department associated with each product.

These categorical relationships will be used to construct customer-level category affinity features.

In [14]:
# Step 2.8 — Enrich product features with aisle and department

product_features = product_features.merge(
    products[
        [
            "product_id",
            "aisle_id",
            "department_id"
        ]
    ],
    on="product_id",
    how="left"
)

print("Product features shape:")
print(product_features.shape)

print("\nProduct features with categories:")
print(product_features.head())

print("\nMissing values:")
print(product_features.isna().sum())

print("\nUnique aisles:")
print(product_features["aisle_id"].nunique())

print("\nUnique departments:")
print(product_features["department_id"].nunique())

Product features shape:
(49677, 6)

Product features with categories:
   product_id  product_total_purchases  product_unique_users  \
0           1                     1852                   716   
1           2                       90                    78   
2           3                      277                    74   
3           4                      329                   182   
4           5                       15                     6   

   product_reorder_rate  aisle_id  department_id  
0              0.613391        61             19  
1              0.133333       104             13  
2              0.732852        94              7  
3              0.446809        38              1  
4              0.600000         5             13  

Missing values:
product_id                 0
product_total_purchases    0
product_unique_users       0
product_reorder_rate       0
aisle_id                   0
department_id              0
dtype: int64

Unique aisles:
134

Unique departm

## Step 2.9 — Create Customer Category Affinity Features

Historical customer-product interactions are combined with product category metadata to measure each customer's purchasing affinity for departments and aisles.

Purchase counts and purchase shares are calculated from historical prior-order behavior only.

In [15]:
# Step 2.9A — Attach product categories to customer-product history

customer_product_categories = customer_product_features[
    [
        "user_id",
        "product_id",
        "user_product_purchase_count"
    ]
].merge(
    product_features[
        [
            "product_id",
            "aisle_id",
            "department_id"
        ]
    ],
    on="product_id",
    how="left"
)

print("Customer-product category table shape:")
print(customer_product_categories.shape)

print("\nSample:")
print(customer_product_categories.head())

print("\nMissing values:")
print(customer_product_categories.isna().sum())

Customer-product category table shape:
(13307953, 5)

Sample:
   user_id  product_id  user_product_purchase_count  aisle_id  department_id
0        7        4920                            7       123              4
1        7        4945                            3       123              4
2        7        8277                            3        24              4
3        7       11520                            1        86             16
4        7       13198                            8       122             12

Missing values:
user_id                        0
product_id                     0
user_product_purchase_count    0
aisle_id                       0
department_id                  0
dtype: int64


In [16]:
# Step 2.9B — Create customer-department affinity features

user_department_features = (
    customer_product_categories
    .groupby(["user_id", "department_id"])
    .agg(
        user_department_purchase_count=(
            "user_product_purchase_count",
            "sum"
        )
    )
    .reset_index()
)

# Total historical purchases for each customer
user_total_purchase_counts = (
    user_department_features
    .groupby("user_id")["user_department_purchase_count"]
    .transform("sum")
)

# Department share of the customer's total historical purchases
user_department_features["user_department_purchase_share"] = (
    user_department_features["user_department_purchase_count"]
    / user_total_purchase_counts
)

print("User-department feature shape:")
print(user_department_features.shape)

print("\nSample:")
print(user_department_features.head(10))

print("\nMissing values:")
print(user_department_features.isna().sum())

print("\nShare statistics:")
print(
    user_department_features[
        "user_department_purchase_share"
    ].describe()
)

User-department feature shape:
(2232789, 4)

Sample:
   user_id  department_id  user_department_purchase_count  \
0        1              4                               5   
1        1              7                              13   
2        1             13                               1   
3        1             14                               3   
4        1             16                              13   
5        1             17                               2   
6        1             19                              22   
7        2              1                              17   
8        2              3                               2   
9        2              4                              36   

   user_department_purchase_share  
0                        0.084746  
1                        0.220339  
2                        0.016949  
3                        0.050847  
4                        0.220339  
5                        0.033898  
6                      

In [17]:
# Step 2.9C — Create customer-aisle affinity features

user_aisle_features = (
    customer_product_categories
    .groupby(["user_id", "aisle_id"])
    .agg(
        user_aisle_purchase_count=(
            "user_product_purchase_count",
            "sum"
        )
    )
    .reset_index()
)

# Total historical purchases for each customer
user_total_aisle_purchases = (
    user_aisle_features
    .groupby("user_id")["user_aisle_purchase_count"]
    .transform("sum")
)

# Aisle share of the customer's total historical purchases
user_aisle_features["user_aisle_purchase_share"] = (
    user_aisle_features["user_aisle_purchase_count"]
    / user_total_aisle_purchases
)

print("User-aisle feature shape:")
print(user_aisle_features.shape)

print("\nSample:")
print(user_aisle_features.head(10))

print("\nMissing values:")
print(user_aisle_features.isna().sum())

print("\nShare statistics:")
print(
    user_aisle_features[
        "user_aisle_purchase_share"
    ].describe()
)

User-aisle feature shape:
(5729249, 4)

Sample:
   user_id  aisle_id  user_aisle_purchase_count  user_aisle_purchase_share
0        1        21                          8                   0.135593
1        1        23                         12                   0.203390
2        1        24                          5                   0.084746
3        1        45                          1                   0.016949
4        1        53                          2                   0.033898
5        1        54                          2                   0.033898
6        1        77                         13                   0.220339
7        1        88                          1                   0.016949
8        1        91                          2                   0.033898
9        1       117                          9                   0.152542

Missing values:
user_id                      0
aisle_id                     0
user_aisle_purchase_count    0
user_aisle_purcha

## Step 2.10 — Attach Category Affinity to Customer-Product Features

For each customer-product pair, the product's department and aisle are used to retrieve the customer's historical purchasing affinity for those categories.

This creates customer-product features that indicate how strongly the customer is associated with the product's department and aisle.

In [18]:
# Step 2.10A — Attach department affinity to customer-product features

# Map each product to its department
product_department_map = product_features[
    ["product_id", "department_id"]
].drop_duplicates("product_id")

# Add product department to customer-product table
customer_product_features = customer_product_features.merge(
    product_department_map,
    on="product_id",
    how="left"
)

# Add customer's affinity for that department
customer_product_features = customer_product_features.merge(
    user_department_features[
        [
            "user_id",
            "department_id",
            "user_department_purchase_count",
            "user_department_purchase_share"
        ]
    ],
    on=["user_id", "department_id"],
    how="left"
)

# Customers with no historical purchases in that department
# receive zero affinity.
customer_product_features[
    [
        "user_department_purchase_count",
        "user_department_purchase_share"
    ]
] = customer_product_features[
    [
        "user_department_purchase_count",
        "user_department_purchase_share"
    ]
].fillna(0)

print("Shape:")
print(customer_product_features.shape)

print("\nDepartment affinity features:")
print(
    customer_product_features[
        [
            "user_id",
            "product_id",
            "department_id",
            "user_department_purchase_count",
            "user_department_purchase_share"
        ]
    ].head(10)
)

print("\nMissing values:")
print(
    customer_product_features[
        [
            "department_id",
            "user_department_purchase_count",
            "user_department_purchase_share"
        ]
    ].isna().sum()
)

Shape:
(13307953, 11)

Department affinity features:
   user_id  product_id  department_id  user_department_purchase_count  \
0        7        4920              4                              57   
1        7        4945              4                              57   
2        7        8277              4                              57   
3        7       11520             16                              32   
4        7       13198             12                               8   
5        7       17638              7                              51   
6        7       27344             20                              13   
7        7       32177              4                              57   
8        7       37602              7                              51   
9        7       40852             16                              32   

   user_department_purchase_share  
0                        0.276699  
1                        0.276699  
2                        0.276699  

In [19]:
# Step 2.10B — Attach aisle affinity to customer-product features

# Map each product to its aisle
product_aisle_map = product_features[
    ["product_id", "aisle_id"]
].drop_duplicates("product_id")

# Add product aisle to customer-product table
customer_product_features = customer_product_features.merge(
    product_aisle_map,
    on="product_id",
    how="left"
)

# Add customer's affinity for that aisle
customer_product_features = customer_product_features.merge(
    user_aisle_features[
        [
            "user_id",
            "aisle_id",
            "user_aisle_purchase_count",
            "user_aisle_purchase_share"
        ]
    ],
    on=["user_id", "aisle_id"],
    how="left"
)

# No historical purchases in the aisle = zero observed affinity
customer_product_features[
    [
        "user_aisle_purchase_count",
        "user_aisle_purchase_share"
    ]
] = customer_product_features[
    [
        "user_aisle_purchase_count",
        "user_aisle_purchase_share"
    ]
].fillna(0)

print("Shape:")
print(customer_product_features.shape)

print("\nAisle affinity features:")
print(
    customer_product_features[
        [
            "user_id",
            "product_id",
            "aisle_id",
            "user_aisle_purchase_count",
            "user_aisle_purchase_share"
        ]
    ].head(10)
)

print("\nMissing values:")
print(
    customer_product_features[
        [
            "aisle_id",
            "user_aisle_purchase_count",
            "user_aisle_purchase_share"
        ]
    ].isna().sum()
)

Shape:
(13307953, 14)

Aisle affinity features:
   user_id  product_id  aisle_id  user_aisle_purchase_count  \
0        7        4920       123                         17   
1        7        4945       123                         17   
2        7        8277        24                         24   
3        7       11520        86                          1   
4        7       13198       122                          8   
5        7       17638        26                         23   
6        7       27344        96                         10   
7        7       32177        24                         24   
8        7       37602        26                         23   
9        7       40852        91                         13   

   user_aisle_purchase_share  
0                   0.082524  
1                   0.082524  
2                   0.116505  
3                   0.004854  
4                   0.038835  
5                   0.111650  
6                   0.048544  
7         

In [21]:
print("Variables available:")
print([name for name in globals() if "affinity" in name.lower()])

Variables available:
[]


In [22]:
print([name for name in globals() if "customer" in name.lower()])

['customer_features', 'customer_product_stats', 'customer_product_features', 'customer_product_categories', 'customer_cols']


In [24]:
print("ALL CUSTOMER/PRODUCT RELATED VARIABLES:")

for name in list(globals().keys()):
    if (
        "customer" in name.lower()
        or "department" in name.lower()
        or "aisle" in name.lower()
        or "product" in name.lower()
    ):
        print(name)

ALL CUSTOMER/PRODUCT RELATED VARIABLES:
products
aisles
departments
customer_features
customer_product_stats
product_id
customer_product_features
product_features
customer_product_categories
user_department_features
user_aisle_features
user_total_aisle_purchases
product_department_map
product_aisle_map
customer_cols
department_cols


## Step 2.11 — Assemble Customer and Product Features

The engineered customer-level and product-level features are merged into the customer-product feature table. The resulting table provides a unified feature representation for each historical customer-product pair.

In [26]:
# ============================================================
# Step 2.11 — Assemble Customer and Product Features
# ============================================================

print("=== STEP 2.11: ASSEMBLE CUSTOMER AND PRODUCT FEATURES ===\n")


# ------------------------------------------------------------
# 1. Clean up customer features if Step 2.11 was previously
#    partially executed
# ------------------------------------------------------------

customer_feature_names = [
    "user_total_orders",
    "user_avg_days_between_orders",
    "user_avg_order_hour",
    "user_avg_order_dow"
]

# If duplicated _x/_y columns exist, keep the _x versions
# and rename them back to their original names.
for col in customer_feature_names:
    x_col = col + "_x"
    y_col = col + "_y"

    if x_col in customer_product_features.columns:
        # Verify that _x and _y contain the same values if both exist
        if y_col in customer_product_features.columns:
            same_values = (
                customer_product_features[x_col]
                .equals(customer_product_features[y_col])
            )

            print(f"{col}: _x and _y identical = {same_values}")

            # Remove duplicate _y column
            customer_product_features.drop(
                columns=[y_col],
                inplace=True
            )

        # Rename _x back to the original feature name
        customer_product_features.rename(
            columns={x_col: col},
            inplace=True
        )


print("\nCleaned customer-product feature shape:")
print(customer_product_features.shape)


# ------------------------------------------------------------
# 2. Add product-level features
# ------------------------------------------------------------

product_cols = [
    "product_id",
    "product_total_purchases",
    "product_unique_users",
    "product_reorder_rate"
]

# Only add product features if they are not already present
existing_product_features = [
    col for col in product_cols
    if col in customer_product_features.columns
]

if len(existing_product_features) > 1:
    print("\nProduct features already present. Skipping product merge.")

else:
    customer_product_features = customer_product_features.merge(
        product_features[product_cols],
        on="product_id",
        how="left"
    )

    print("\nAfter adding product features:")
    print(customer_product_features.shape)


# ------------------------------------------------------------
# 3. Display final columns
# ------------------------------------------------------------

print("\nFinal feature columns:")
print(customer_product_features.columns.tolist())


# ------------------------------------------------------------
# 4. Check missing values
# ------------------------------------------------------------

print("\nMissing values:")
print(customer_product_features.isna().sum())


# ------------------------------------------------------------
# 5. Check duplicate customer-product pairs
# ------------------------------------------------------------

duplicate_count = customer_product_features.duplicated(
    subset=["user_id", "product_id"]
).sum()

print("\nDuplicate customer-product pairs:")
print(duplicate_count)


# ------------------------------------------------------------
# 6. Display sample
# ------------------------------------------------------------

print("\nSample of final feature dataset:")
print(customer_product_features.head())


# ------------------------------------------------------------
# 7. Final shape
# ------------------------------------------------------------

print("\nFinal assembled feature shape:")
print(customer_product_features.shape)

=== STEP 2.11: ASSEMBLE CUSTOMER AND PRODUCT FEATURES ===

user_total_orders: _x and _y identical = True
user_avg_days_between_orders: _x and _y identical = True
user_avg_order_hour: _x and _y identical = True
user_avg_order_dow: _x and _y identical = True

Cleaned customer-product feature shape:
(13307953, 21)

Product features already present. Skipping product merge.

Final feature columns:
['user_id', 'product_id', 'user_product_purchase_count', 'user_product_reorder_count', 'user_product_last_order_number', 'user_product_reorder_rate', 'user_product_avg_cart_position', 'user_product_recency_orders', 'department_id', 'user_department_purchase_count', 'user_department_purchase_share', 'aisle_id', 'user_aisle_purchase_count', 'user_aisle_purchase_share', 'user_total_orders', 'user_avg_days_between_orders', 'user_avg_order_hour', 'user_avg_order_dow', 'product_total_purchases', 'product_unique_users', 'product_reorder_rate']

Missing values:
user_id                           0
product_

## Step 2.12 — Feature Quality Validation

The engineered feature dataset is validated for duplicate customer-product pairs, missing values, invalid numerical values, and feature ranges. These checks ensure that the feature pipeline produces consistent and reliable data before the dataset is stored and used in later ML stages.

In [27]:
# ============================================================
# Step 2.12 — Feature Quality Validation
# ============================================================

print("=== FEATURE DATASET VALIDATION ===\n")


# ------------------------------------------------------------
# 1. Shape
# ------------------------------------------------------------
print("Shape:")
print(customer_product_features.shape)


# ------------------------------------------------------------
# 2. Duplicate customer-product pairs
# ------------------------------------------------------------
duplicate_count = customer_product_features.duplicated(
    subset=["user_id", "product_id"]
).sum()

print("\nDuplicate customer-product pairs:")
print(duplicate_count)


# ------------------------------------------------------------
# 3. Missing values
# ------------------------------------------------------------
print("\nMissing values:")
missing_values = customer_product_features.isna().sum()
print(missing_values)


# ------------------------------------------------------------
# 4. Check numerical columns
# ------------------------------------------------------------
numeric_columns = customer_product_features.select_dtypes(
    include=["number"]
).columns

print("\nNumeric columns:")
print(numeric_columns.tolist())


# ------------------------------------------------------------
# 5. Check for infinite values
# ------------------------------------------------------------
infinite_counts = (
    customer_product_features[numeric_columns]
    .isin([float("inf"), float("-inf")])
    .sum()
)

print("\nInfinite values:")
print(infinite_counts)


# ------------------------------------------------------------
# 6. Validate rate/share features
# ------------------------------------------------------------
rate_features = [
    "user_product_reorder_rate",
    "user_department_purchase_share",
    "user_aisle_purchase_share",
    "product_reorder_rate"
]

print("\nRate/share range checks:")

for col in rate_features:
    print(
        f"{col}: "
        f"min={customer_product_features[col].min():.4f}, "
        f"max={customer_product_features[col].max():.4f}"
    )


# ------------------------------------------------------------
# 7. Check non-negative count features
# ------------------------------------------------------------
count_features = [
    "user_product_purchase_count",
    "user_product_reorder_count",
    "user_product_last_order_number",
    "user_product_avg_cart_position",
    "user_product_recency_orders",
    "user_department_purchase_count",
    "user_aisle_purchase_count",
    "user_total_orders",
    "product_total_purchases",
    "product_unique_users"
]

print("\nNegative value checks:")

for col in count_features:
    negative_count = (
        customer_product_features[col] < 0
    ).sum()

    print(f"{col}: {negative_count}")


# ------------------------------------------------------------
# 8. Final data types
# ------------------------------------------------------------
print("\nData types:")
print(customer_product_features.dtypes)


# ------------------------------------------------------------
# 9. Final validation summary
# ------------------------------------------------------------
print("\n=== VALIDATION SUMMARY ===")

print(
    "Shape:",
    customer_product_features.shape
)

print(
    "Duplicate customer-product pairs:",
    duplicate_count
)

print(
    "Total missing values:",
    missing_values.sum()
)

print(
    "Total infinite values:",
    infinite_counts.sum()
)

print("\nFeature validation completed.")

=== FEATURE DATASET VALIDATION ===

Shape:
(13307953, 21)

Duplicate customer-product pairs:
0

Missing values:
user_id                           0
product_id                        0
user_product_purchase_count       0
user_product_reorder_count        0
user_product_last_order_number    0
user_product_reorder_rate         0
user_product_avg_cart_position    0
user_product_recency_orders       0
department_id                     0
user_department_purchase_count    0
user_department_purchase_share    0
aisle_id                          0
user_aisle_purchase_count         0
user_aisle_purchase_share         0
user_total_orders                 0
user_avg_days_between_orders      0
user_avg_order_hour               0
user_avg_order_dow                0
product_total_purchases           0
product_unique_users              0
product_reorder_rate              0
dtype: int64

Numeric columns:
['user_id', 'product_id', 'user_product_purchase_count', 'user_product_reorder_count', 'user_product_